In [1]:
import pandas as pd
import requests
import os
import time

In [2]:
url = "https://solarcentre.spinifexvalley.com.au/export/73-Site_DKA-M19_A-Phase.csv"

r = requests.get(url)

with open("dkasc35.csv", "wb") as f:
    f.write(r.content)

print("Downloaded:", len(r.content) / 1e6, "MB")

Downloaded: 259.499475 MB


In [3]:
dkasc35  = pd.read_csv('dkasc35.csv')

In [4]:
dkasc35

,timestamp,Active_Energy_Delivered_Received,Current_Phase_Average,Active_Power,Performance_Ratio,Wind_Speed,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,Diffuse_Horizontal_Radiation,Wind_Direction,Weather_Daily_Rainfall,Radiation_Global_Tilted,Radiation_Diffuse_Tilted
0,2010-02-18 14:35:00,0.0,0.000000,0.000000,NaN,6.793873,35.132046,13.933495,1000.515625,97.682610,126.266418,0.0,NaN,NaN
1,2010-02-18 14:40:00,0.0,0.000000,0.000000,NaN,6.926013,34.586330,14.363612,989.110413,102.564949,116.272385,0.0,NaN,NaN
2,2010-02-18 14:45:00,0.0,0.000000,0.000000,NaN,6.824874,34.628662,13.933328,977.882629,102.709160,141.693970,0.0,NaN,NaN
3,2010-02-18 14:50:00,0.0,0.000000,0.000000,NaN,5.291194,35.258572,13.457552,963.508484,100.324097,130.381912,0.0,NaN,NaN
4,2010-02-18 14:55:00,0.0,0.000000,0.000000,NaN,6.065388,35.220058,13.886837,939.744995,105.697617,126.441544,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1619052,2025-09-04 11:45:00,112346.0,16.530003,4.027533,63.913963,NaN,24.142389,10.063275,1001.720337,85.369202,32.207203,0.0,1145.725586,103.592773
1619053,2025-09-04 11:50:00,112346.0,16.405331,3.995933,65.686432,NaN,24.424450,10.260159,1002.639893,81.787186,32.197113,0.0,1106.062988,103.076294
1619054,2025-09-04 11:55:00,NaN,NaN,NaN,NaN,NaN,24.469185,10.172641,1011.761597,86.959572,32.198853,0.0,NaN,NaN
1619055,2025-09-04 12:00:00,112347.0,16.642670,4.053600,71.596481,NaN,24.084011,10.374229,1018.724426,90.641022,32.169468,0.0,1029.405518,106.023285


In [5]:
df = pd.read_csv('dkasc35.csv')

#clearing discrepancies
df.iloc[1617509] = ['2025-08-30 03:10:00', 112173.0, 0.3, 0.0, 0.0,0.0,10.821783, 47.015762,1.028532,0.620103,34.234615,0.0,3.238491,0.412667]
#print(df.iloc[1617509])

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.set_index("timestamp")

# Define aggregation rules
agg_dict = {
    'Active_Energy_Delivered_Received': 'sum',
    'Current_Phase_Average': 'mean',
    'Active_Power': 'mean',
    'Performance_Ratio': 'mean',
    'Wind_Speed': 'mean',
    'Weather_Temperature_Celsius': 'mean',
    'Weather_Relative_Humidity': 'mean',
    'Global_Horizontal_Radiation': 'mean',
    'Diffuse_Horizontal_Radiation': 'mean',
    'Wind_Direction': 'mean',
    'Weather_Daily_Rainfall': 'max',
    'Radiation_Global_Tilted': 'mean',
    'Radiation_Diffuse_Tilted': 'mean'
}

# Resample to hourly
df_hourly = df.resample('1h').agg(agg_dict).reset_index()

# Save
df_hourly.to_csv('hourly_data.csv', index=False)

In [6]:
print(df_hourly.shape)
print(df_hourly.head())
print(df_hourly.tail())
print(df_hourly.isna().sum())
print(df_hourly["timestamp"].min())
print(df_hourly["timestamp"].max())

(136247, 14)
            timestamp  Active_Energy_Delivered_Received  \
0 2010-02-18 14:00:00                               0.0   
1 2010-02-18 15:00:00                               0.0   
2 2010-02-18 16:00:00                               0.0   
3 2010-02-18 17:00:00                               0.0   
4 2010-02-18 18:00:00                               0.0   

   Current_Phase_Average  Active_Power  Performance_Ratio  Wind_Speed  \
0                    0.0           0.0                NaN    6.380269   
1                    0.0           0.0                NaN    5.347911   
2                    0.0           0.0                NaN    4.944388   
3                    0.0           0.0                NaN    5.295506   
4                    0.0           0.0                NaN    5.150523   

   Weather_Temperature_Celsius  Weather_Relative_Humidity  \
0                    34.965134                  13.914965   
1                    35.096028                  13.318755   
2         

In [7]:
# 1. Check energy/power behaviour
print(df_hourly[
    ['timestamp',
     'Active_Energy_Delivered_Received',
     'Active_Power']
].head(30))


# 2. Check hourly gaps
time_diff = df_hourly['timestamp'].diff()
print("\nTime differences:")
print(time_diff.value_counts().head(10))


# 3. Check total missing hours
expected_hours = (
    df_hourly['timestamp'].max() -
    df_hourly['timestamp'].min()
).total_seconds() / 3600 + 1

actual_hours = len(df_hourly)

print("\nExpected hours:", expected_hours)
print("Actual hours:", actual_hours)
print("Missing hours:", expected_hours - actual_hours)

             timestamp  Active_Energy_Delivered_Received  Active_Power
0  2010-02-18 14:00:00                               0.0           0.0
1  2010-02-18 15:00:00                               0.0           0.0
2  2010-02-18 16:00:00                               0.0           0.0
3  2010-02-18 17:00:00                               0.0           0.0
4  2010-02-18 18:00:00                               0.0           0.0
5  2010-02-18 19:00:00                               0.0           0.0
6  2010-02-18 20:00:00                               0.0           0.0
7  2010-02-18 21:00:00                               0.0           0.0
8  2010-02-18 22:00:00                               0.0           0.0
9  2010-02-18 23:00:00                               0.0           0.0
10 2010-02-19 00:00:00                               0.0           0.0
11 2010-02-19 01:00:00                               0.0           0.0
12 2010-02-19 02:00:00                               0.0           0.0
13 201

In [8]:
# Read the hourly dataset
df = pd.read_csv("hourly_data.csv")

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Create output folder
os.makedirs("hourly_by_year", exist_ok=True)

# Split by year
for year, group in df.groupby(df["timestamp"].dt.year):
    filename = f"hourly_by_year/hourly_data_{year}.csv"
    group.to_csv(filename, index=False)
    print(f"{year}: {len(group)} rows → {filename}")

2010: 7594 rows → hourly_by_year/hourly_data_2010.csv
2011: 8760 rows → hourly_by_year/hourly_data_2011.csv
2012: 8784 rows → hourly_by_year/hourly_data_2012.csv
2013: 8760 rows → hourly_by_year/hourly_data_2013.csv
2014: 8760 rows → hourly_by_year/hourly_data_2014.csv
2015: 8760 rows → hourly_by_year/hourly_data_2015.csv
2016: 8784 rows → hourly_by_year/hourly_data_2016.csv
2017: 8760 rows → hourly_by_year/hourly_data_2017.csv
2018: 8760 rows → hourly_by_year/hourly_data_2018.csv
2019: 8760 rows → hourly_by_year/hourly_data_2019.csv
2020: 8784 rows → hourly_by_year/hourly_data_2020.csv
2021: 8760 rows → hourly_by_year/hourly_data_2021.csv
2022: 8760 rows → hourly_by_year/hourly_data_2022.csv
2023: 8760 rows → hourly_by_year/hourly_data_2023.csv
2024: 8784 rows → hourly_by_year/hourly_data_2024.csv
2025: 5917 rows → hourly_by_year/hourly_data_2025.csv


In [9]:
import shutil

shutil.make_archive(
    "hourly_data_by_year",
    "zip",
    "hourly_by_year"
)

print("Created hourly_data_by_year.zip")

Created hourly_data_by_year.zip


In [10]:
# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

DKASC_FOLDER = "hourly_by_year"
OUTPUT_FOLDER = "merged_by_year"

LATITUDE = -23.6980
LONGITUDE = 133.8807

NASA_PARAMETERS = "ALLSKY_SFC_SW_DWN,T2M,RH2M,WS10M"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# --------------------------------------------------
# FUNCTION TO GET NASA DATA
# --------------------------------------------------

def get_nasa_data(year):

    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"

    params = {
        "parameters": NASA_PARAMETERS,
        "community": "RE",
        "longitude": LONGITUDE,
        "latitude": LATITUDE,
        "start": f"{year}0101",
        "end": f"{year}1231",
        "format": "JSON"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception(
            f"NASA API failed for {year}: "
            f"HTTP {response.status_code}"
        )

    data = response.json()

    parameters = data["properties"]["parameter"]

    nasa = pd.DataFrame(parameters)

    nasa.index = pd.to_datetime(
        nasa.index,
        format="%Y%m%d%H"
    )

    nasa.index.name = "timestamp"

    nasa = nasa.reset_index()

    return nasa


# --------------------------------------------------
# PROCESS EACH YEAR
# --------------------------------------------------

results = []

for year in range(2010, 2026):

    print("\n" + "=" * 50)
    print(f"Processing {year}")
    print("=" * 50)

    dkasc_file = (
        f"{DKASC_FOLDER}/hourly_data_{year}.csv"
    )

    output_file = (
        f"{OUTPUT_FOLDER}/merged_{year}.csv"
    )

    # Check if DKASC file exists
    if not os.path.exists(dkasc_file):
        print(f"DKASC file not found: {dkasc_file}")
        continue

    # ----------------------------------------------
    # Load DKASC
    # ----------------------------------------------

    dkasc = pd.read_csv(dkasc_file)

    dkasc["timestamp"] = pd.to_datetime(
        dkasc["timestamp"]
    )

    dkasc = dkasc.sort_values("timestamp")

    print(f"DKASC rows: {len(dkasc)}")


    # ----------------------------------------------
    # Get NASA
    # ----------------------------------------------

    try:

        nasa = get_nasa_data(year)

        print(f"NASA rows: {len(nasa)}")

    except Exception as e:

        print(f"NASA ERROR: {e}")
        continue


    # ----------------------------------------------
    # Merge
    # ----------------------------------------------

    merged = pd.merge(
        dkasc,
        nasa,
        on="timestamp",
        how="left"
    )

    print(f"Merged rows: {len(merged)}")


    # ----------------------------------------------
    # Check NASA missing values
    # ----------------------------------------------

    nasa_columns = [
        "ALLSKY_SFC_SW_DWN",
        "T2M",
        "RH2M",
        "WS10M"
    ]

    missing = merged[nasa_columns].isna().sum()

    print("NASA missing values:")
    print(missing)


    # ----------------------------------------------
    # Save
    # ----------------------------------------------

    merged.to_csv(
        output_file,
        index=False
    )

    print(f"Saved: {output_file}")


    results.append({
        "year": year,
        "dkasc_rows": len(dkasc),
        "nasa_rows": len(nasa),
        "merged_rows": len(merged),
        "nasa_missing": merged[nasa_columns].isna().sum().sum()
    })


    # Avoid hitting API too quickly
    time.sleep(1)


# --------------------------------------------------
# SUMMARY
# --------------------------------------------------

summary = pd.DataFrame(results)

print("\n\nFINAL SUMMARY")
print(summary)


Processing 2010
DKASC rows: 7594
NASA rows: 8760
Merged rows: 7594
NASA missing values:
ALLSKY_SFC_SW_DWN    0
T2M                  0
RH2M                 0
WS10M                0
dtype: int64
Saved: merged_by_year/merged_2010.csv

Processing 2011
DKASC rows: 8760
NASA rows: 8760
Merged rows: 8760
NASA missing values:
ALLSKY_SFC_SW_DWN    0
T2M                  0
RH2M                 0
WS10M                0
dtype: int64
Saved: merged_by_year/merged_2011.csv

Processing 2012
DKASC rows: 8784
NASA rows: 8784
Merged rows: 8784
NASA missing values:
ALLSKY_SFC_SW_DWN    0
T2M                  0
RH2M                 0
WS10M                0
dtype: int64
Saved: merged_by_year/merged_2012.csv

Processing 2013
DKASC rows: 8760
NASA rows: 8760
Merged rows: 8760
NASA missing values:
ALLSKY_SFC_SW_DWN    0
T2M                  0
RH2M                 0
WS10M                0
dtype: int64
Saved: merged_by_year/merged_2013.csv

Processing 2014
DKASC rows: 8760
NASA rows: 8760
Merged rows: 8760
NASA

In [11]:
import glob

files = glob.glob("merged_by_year/merged_*.csv")

all_data = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

all_data["timestamp"] = pd.to_datetime(
    all_data["timestamp"]
)

all_data = all_data.sort_values("timestamp")

all_data.to_csv(
    "PV_NASA_combined.csv",
    index=False
)

print("Final shape:", all_data.shape)
print(all_data.head())
print(all_data.tail())

Final shape: (136247, 18)
                timestamp  Active_Energy_Delivered_Received  \
67309 2010-02-18 14:00:00                               0.0   
67310 2010-02-18 15:00:00                               0.0   
67311 2010-02-18 16:00:00                               0.0   
67312 2010-02-18 17:00:00                               0.0   
67313 2010-02-18 18:00:00                               0.0   

       Current_Phase_Average  Active_Power  Performance_Ratio  Wind_Speed  \
67309                    0.0           0.0                NaN    6.380269   
67310                    0.0           0.0                NaN    5.347911   
67311                    0.0           0.0                NaN    4.944388   
67312                    0.0           0.0                NaN    5.295506   
67313                    0.0           0.0                NaN    5.150523   

       Weather_Temperature_Celsius  Weather_Relative_Humidity  \
67309                    34.965134                  13.914965   
67

In [12]:
print(
    all_data[
        [
            "timestamp",
            "Active_Energy_Delivered_Received",
            "Active_Power"
        ]
    ].loc[
        all_data["Active_Power"] > 0
    ].head(20)
)

                 timestamp  Active_Energy_Delivered_Received  Active_Power
118182 2014-12-09 07:00:00                               0.0      0.650000
118183 2014-12-09 08:00:00                              14.0      2.120694
118184 2014-12-09 09:00:00                              46.0      2.862194
118185 2014-12-09 10:00:00                              85.0      3.679589
118186 2014-12-09 11:00:00                             134.0      4.391619
118187 2014-12-09 12:00:00                             188.0      4.564994
118188 2014-12-09 13:00:00                             237.0      3.561603
118189 2014-12-09 14:00:00                             282.0      3.695561
118190 2014-12-09 15:00:00                             313.0      2.473275
118191 2014-12-09 16:00:00                             338.0      1.231419
118192 2014-12-09 17:00:00                             348.0      0.730453
118193 2014-12-09 18:00:00                             360.0      0.201931
118194 2014-12-09 19:00:0

In [13]:
all_data["energy_from_power"] = (
    all_data["Active_Power"] * 1
)

print(
    all_data[
        [
            "timestamp",
            "Active_Energy_Delivered_Received",
            "Active_Power",
            "energy_from_power"
        ]
    ].loc[
        all_data["Active_Power"] > 0
    ].head(20)
)

                 timestamp  Active_Energy_Delivered_Received  Active_Power  \
118182 2014-12-09 07:00:00                               0.0      0.650000   
118183 2014-12-09 08:00:00                              14.0      2.120694   
118184 2014-12-09 09:00:00                              46.0      2.862194   
118185 2014-12-09 10:00:00                              85.0      3.679589   
118186 2014-12-09 11:00:00                             134.0      4.391619   
118187 2014-12-09 12:00:00                             188.0      4.564994   
118188 2014-12-09 13:00:00                             237.0      3.561603   
118189 2014-12-09 14:00:00                             282.0      3.695561   
118190 2014-12-09 15:00:00                             313.0      2.473275   
118191 2014-12-09 16:00:00                             338.0      1.231419   
118192 2014-12-09 17:00:00                             348.0      0.730453   
118193 2014-12-09 18:00:00                             360.0    